# Olist E-Commerce Data Analysis
**Objective:** Perform End-to-End Data Preparation. This includes Data Loading, Exploratory Data Analysis (EDA), Data Cleaning, exporting relational tables for SQL Server, and conducting Advanced Analytics (RFM Segmentation & Cohort Analysis) for Power BI.

In [17]:
import pandas as pd
import numpy as np

## 1. Data Loading
Importing all the necessary raw datasets provided by Olist.

In [18]:
# Loading all datasets
orders = pd.read_csv('olist_orders_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')

Based on business logic, we perform the following transformations:
1. **Products:** Merge the translation table to convert Portuguese categories to English. Impute missing categories with 'Unknown' and missing numerical specs with 0.
2. **Orders:** - Drop 8 anomalous rows where status is 'delivered' but delivery date is missing.
   - Impute missing 'order_approved_at' values with the 'order_purchase_timestamp' assuming instant payment approval.
3. **Reviews:** Fill missing text reviews with 'No Title' and 'No Comment' to maintain data integrity.

In [19]:
# 1. Products Translation & Cleaning
# Step 1: Merge translation table to convert Portuguese categories to English
products = pd.merge(products, translation, on='product_category_name', how='left')
products.drop(columns=['product_category_name'], inplace=True)
products.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)

# Step 2: Impute missing categories with 'Unknown'
products['product_category'] = products['product_category'].fillna('Unknown')

# Step 3: Impute missing numerical specifications with 0
numeric_cols = [
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'
]
for col in numeric_cols:
    products[col] = products[col].fillna(0)

In [20]:
# 2. Orders Cleaning
# Drop anomalous delivered orders with missing delivery dates
orders = orders[~((orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull()))]

# Impute missing approval dates with purchase timestamps
orders['order_approved_at'] = orders['order_approved_at'].fillna(orders['order_purchase_timestamp'])

In [21]:
# 3. Reviews Cleaning
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Comment')

## 3. Data Quality Summary (EDA)
A quick loop to inspect the shape and missing values of our cleaned datasets before exporting.

In [22]:
datasets = {
    'Orders': orders, 'Products': products, 'Customers': customers, 
    'Payments': payments, 'Items': items, 'Reviews': reviews, 'Sellers': sellers
}

print("Cleaned Data Quality Summary")
for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns | Total Nulls: {missing}")

Cleaned Data Quality Summary
Orders: 99433 rows, 8 columns | Total Nulls: 4739
Products: 32951 rows, 9 columns | Total Nulls: 0
Customers: 99441 rows, 5 columns | Total Nulls: 0
Payments: 103886 rows, 5 columns | Total Nulls: 0
Items: 112650 rows, 7 columns | Total Nulls: 0
Reviews: 99224 rows, 7 columns | Total Nulls: 0
Sellers: 3095 rows, 4 columns | Total Nulls: 0


## 4. Exporting Data for SQL Server (Star Schema Modeling)
Saving the cleaned relational tables as flat files ready for ingestion into SQL Server.

In [23]:
orders.to_csv('sql_orders.csv', index=False)
products.to_csv('sql_products.csv', index=False)
customers.to_csv('sql_customers.csv', index=False)
payments.to_csv('sql_payments.csv', index=False)
items.to_csv('sql_items.csv', index=False)
reviews.to_csv('sql_reviews.csv', index=False)
sellers.to_csv('sql_sellers.csv', index=False)
print("All relational tables exported successfully for SQL Server.")

All relational tables exported successfully for SQL Server.


In [24]:
# Checking Data Types for SQL Server Preparation
print("--- Data Types Mapping for SQL Import ---")
for name, df in datasets.items():
    print(f"\n📊 Table: {name}")
    print(df.dtypes)

--- Data Types Mapping for SQL Import ---

📊 Table: Orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

📊 Table: Products
product_id                        str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
product_category                  str
dtype: object

📊 Table: Customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

📊 Table: Payments
order_id                    str
p

## 5. Advanced Analytics: RFM Segmentation
Calculating Recency, Frequency, and Monetary values to segment customers based on purchasing behavior.

In [25]:
df_rfm = orders.merge(customers, on='customer_id').merge(payments, on='order_id')
df_rfm['order_purchase_timestamp'] = pd.to_datetime(df_rfm['order_purchase_timestamp'])
snapshot_date = df_rfm['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df_rfm.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'payment_value': 'sum'
}).reset_index()

rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm['RFM_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)

def assign_segment(score):
    if score >= 12: return 'Champions'
    elif score >= 9: return 'Loyal Customers'
    elif score >= 6: return 'At Risk'
    else: return 'Lost Customers'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

rfm.to_csv('rfm_segmentation.csv', index=False)

rfm[['customer_unique_id', 'Segment', 'RFM_Score']].head()

,customer_unique_id,Segment,RFM_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,Loyal Customers,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,At Risk,6
2,0000f46a3911fa3c0805444483337064,Lost Customers,4
3,0000f6ccb0745a6a4b88665a16c9f078,Lost Customers,4
4,0004aac84e0df4da2b147fca70cf8255,At Risk,7


## 6. Advanced Analytics: Cohort Analysis (Retention)
Creating a retention matrix to analyze customer loyalty over time.

In [26]:
df_cohort = orders.merge(customers, on='customer_id')[['customer_unique_id', 'order_id', 'order_purchase_timestamp']].drop_duplicates()
df_cohort['order_purchase_timestamp'] = pd.to_datetime(df_cohort['order_purchase_timestamp'])

df_cohort['order_month'] = df_cohort['order_purchase_timestamp'].dt.to_period('M')
df_cohort['cohort_month'] = df_cohort.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min').dt.to_period('M')

df_cohort['years_diff'] = df_cohort['order_month'].dt.year - df_cohort['cohort_month'].dt.year
df_cohort['months_diff'] = df_cohort['order_month'].dt.month - df_cohort['cohort_month'].dt.month
df_cohort['cohort_index'] = df_cohort['years_diff'] * 12 + df_cohort['months_diff'] + 1

cohort_data = df_cohort.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().reset_index()
cohort_counts = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_unique_id')

cohort_sizes = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_sizes, axis=0)

retention.iloc[:12, :12].round(3) * 100

cohort_index,1,2,3,4,5,6,7,8,9,10,11,12
cohort_month,,,,,,,,,,,,
2016-09,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,100.0,NaN,NaN,NaN,NaN,NaN,0.3,NaN,NaN,0.3,NaN,0.3
2016-12,100.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,100.0,0.4,0.3,0.1,0.4,0.1,0.5,0.1,0.1,NaN,0.4,0.1
2017-02,100.0,0.2,0.3,0.1,0.4,0.1,0.2,0.2,0.2,0.2,0.1,0.3
2017-03,100.0,0.5,0.4,0.4,0.3,0.2,0.2,0.3,0.3,0.1,0.4,0.2
2017-04,100.0,0.6,0.2,0.2,0.3,0.3,0.3,0.3,0.3,0.2,0.3,0.1
2017-05,100.0,0.5,0.5,0.4,0.3,0.3,0.4,0.2,0.3,0.3,0.3,0.3
2017-06,100.0,0.5,0.4,0.4,0.3,0.4,0.4,0.2,0.1,0.2,0.3,0.4


## 7. Exporting Analytics for Power BI
Saving the final analytical models to be visualized in Power BI dashboards.

In [27]:
rfm.to_csv('rfm_segmentation.csv', index=False)
cohort_counts.to_csv('cohort_retention.csv', index=False)
print("Advanced analytics exported successfully for Power BI.")

Advanced analytics exported successfully for Power BI.
